# 08 · Диагностика обобщения CNN
Исследование после внешнего теста. Порог 0.5 фиксирован; диагностические преобразования не являются новым независимым тестом. Запускайте длительные этапы вручную.

In [ ]:
# 1. Проект и зависимости
import subprocess
import sys
from pathlib import Path

repo=Path('/content/audio-deepfake-detector')
if not (repo/'.git').exists(): subprocess.run(['git','clone','https://github.com/Unvand/audio-deepfake-detector.git',str(repo)],check=True)
else: subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'requirements-colab-train.txt')],check=True)
sys.path.insert(0,str(repo))
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
print('Git commit:',commit)

In [ ]:
# 2. Google Drive
from google.colab import drive

drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/VKR_RuASD')
prepared=base/'prepared_v1'
run=base/'training_v1/runs/baseline_cached_v1'
test=base/'evaluation_v1/baseline_cached_v1'
external=base/'external_test_v1'
previous=external/'results/run_20261127_165511_c1b6379c'
for p in (prepared,run,test,external/'input/REAL',external/'input/FAKE',previous):
 if not p.exists(): raise FileNotFoundError(f'Не найдены данные: {p}')
print('Все каталоги найдены')

In [ ]:
# 3. Проверка протокола и модели
import json

import numpy as np
import pandas as pd
from IPython.display import display

from audio_deepfake_detector.src.colab_cache import _sha256
from audio_deepfake_detector.src.external_evaluation import scan_external_files
from audio_deepfake_detector.src.ruasd_evaluation import load_best_model

checkpoint=run/'best_model.pth'
model,mel,threshold,saved=load_best_model(checkpoint,expected_epoch=8)
test_cfg=json.loads((test/'evaluation_config.json').read_text())
ext_cfg=json.loads((previous/'evaluation_config.json').read_text())
prep_cfg=json.loads((prepared/'preprocessing_config.json').read_text())
checks={
 'Checkpoint SHA-256': _sha256(checkpoint)==test_cfg['checkpoint_sha256']==ext_cfg['checkpoint_sha256'],
 'MelConfig': test_cfg['mel_config']==ext_cfg['mel_config']==saved['mel_config'],
 'REAL=0, FAKE=1': saved['class_mapping']=={'real':0,'fake':1},
 'Сегменты 4 с / 32 кГц': prep_cfg['segment_seconds']==4 and prep_cfg['target_sample_rate']==mel.sample_rate,
 'Порог': threshold==test_cfg['threshold']==ext_cfg['threshold'],
 'Агрегация средним': 'mean' in test_cfg['aggregation'] and 'mean' in ext_cfg['aggregation'],
}
display(pd.DataFrame([{'Проверка':k,'Результат':v} for k,v in checks.items()]))
if not all(checks.values()): raise ValueError('Конфигурации несовместимы')
files,errors=scan_external_files(external)
if not errors.empty: display(errors)
assert len(files)==8
source=pd.read_csv(previous/'file_predictions.csv')
segments=pd.read_csv(previous/'segment_predictions.csv')
assert len(source)==8 and set(source.source_file_id)==set(files.source_file_id)
assert np.allclose(segments.groupby('source_file_id').fake_score.mean().sort_index(),source.set_index('source_file_id').fake_score.sort_index())
print('Прогнозы и агрегация предыдущего запуска проверены')

In [ ]:
# 4. Размер выборки и настройки
import ipywidgets as widgets

sample_size=widgets.IntSlider(value=50,min=5,max=200,step=5,description='На группу')
seed=widgets.IntText(value=42,description='Seed')
display(sample_size,seed)
print('RuASD: стратифицированная выборка подготовленных сегментов; внешние файлы: все 8')

In [ ]:
# 5. Технические признаки (длительная ячейка)
from datetime import UTC

from tqdm.auto import tqdm

from audio_deepfake_detector.src.generalization_diagnostics import (
 anonymous_id,
 inspect_file,
)

RUN_AUDIO_FEATURES=False
if RUN_AUDIO_FEATURES:
 manifest=pd.read_csv(prepared/'ruasd_segments_manifest.csv',low_memory=False)
 sampled=pd.concat([g.sample(n=min(len(g),sample_size.value),random_state=seed.value) for _,g in manifest.groupby('label')],ignore_index=True)
 records=[]
 for row in tqdm(sampled.itertuples(index=False),total=len(sampled),desc='RuASD'):
  values=inspect_file(prepared/row.audio_path,'RuASD',row.label.upper(),anonymous_id(row.segment_id),valid_seconds=getattr(row,'valid_duration_sec',None))
  records.append(values)
 for row in tqdm(files.itertuples(index=False),total=len(files),desc='External'):
  records.append(inspect_file(Path(row.audio_path),'External',row.true_class,row.source_file_id))
 features=pd.DataFrame(records)
 display(features.groupby(['domain','label'])[['rms','silence_fraction','padding_fraction','centroid_hz','flatness']].median())

In [ ]:
# 6. Тишина и padding
RUN_SILENCE=False
if RUN_SILENCE:
 if 'features' not in globals(): raise RuntimeError('Сначала ячейка 5')
 import matplotlib.pyplot as plt
 fig,ax=plt.subplots(1,2,figsize=(10,4),layout='constrained')
 for key,g in features.groupby(['domain','label']):
  ax[0].hist(g.silence_fraction,alpha=.4,label=str(key));ax[1].hist(g.padding_fraction,alpha=.4,label=str(key))
 ax[0].set(title='Доля тишины',xlabel='Доля отсчётов ниже -40 дБ');ax[1].set(title='Доля дополнения нулями',xlabel='Доля')
 for a in ax:a.legend()
 display(fig)

In [ ]:
# 7. Сопоставимые log-Mel примеры и спектральные признаки
RUN_SPECTRA=False
if RUN_SPECTRA:
 if 'sampled' not in globals(): raise RuntimeError('Сначала ячейка 5')
 import tempfile

 import torch
 from torchaudio.transforms import AmplitudeToDB, MelSpectrogram

 from audio_deepfake_detector.src.external_evaluation import decode_audio
 from audio_deepfake_detector.src.ruasd_preparation import (
  resample_mono,
  segment_with_padding,
 )
 transform=MelSpectrogram(sample_rate=mel.sample_rate,n_fft=mel.n_fft,win_length=mel.win_length,hop_length=mel.hop_length,n_mels=mel.n_mels,f_min=mel.f_min,f_max=mel.f_max,power=mel.power)
 db=AmplitudeToDB(stype='power',top_db=80)
 examples=[(f'RuASD {label.upper()}',prepared/g.iloc[0].audio_path) for label,g in sampled.groupby('label')]+[(f'External {label}',Path(g.iloc[0].audio_path)) for label,g in files.groupby('true_class')]
 images=[]
 for title,path in examples:
  x,rate=decode_audio(path);x=resample_mono(x,rate,mel.sample_rate);part=segment_with_padding(x,mel.sample_rate,4)[0][0]
  images.append((title,db(transform(torch.from_numpy(part))).numpy()))
 low=min(a.min() for _,a in images);high=max(a.max() for _,a in images)
 fig,axes=plt.subplots(2,2,figsize=(11,6),layout='constrained')
 for ax,(title,image) in zip(axes.flat,images):ax.imshow(image,origin='lower',aspect='auto',vmin=low,vmax=high);ax.set(title=title,xlabel='Кадры',ylabel='Mel-полоса')
 display(fig)
 print('Спектрограмма сама по себе не устанавливает причину ошибки')

In [ ]:
# 8. Поведение CNN по всем внешним сегментам
RUN_SCORES=False
if RUN_SCORES:
 group=segments.groupby('source_file_id').agg(mean_score=('fake_score','mean'),median_score=('fake_score','median'),fake_segment_fraction=('predicted_label','mean'),n_segments=('segment_id','size'))
 group['id']=['Файл '+str(i+1) for i in range(len(group))]
 display(group.drop(columns=[]).set_index('id'))
 print('Ошибки:',len(source.loc[~source.correct]),'из 8; REAL→FAKE:',len(source[(source.true_label==0)&(~source.correct)]),'FAKE→REAL:',len(source[(source.true_label==1)&(~source.correct)]))

In [ ]:
# 9. Контролируемые преобразования только на временных копиях
RUN_TRANSFORMS=False
if RUN_TRANSFORMS:
 import tempfile

 import soundfile as sf

 from audio_deepfake_detector.src.external_evaluation import (
  decode_audio,
  evaluate_external,
 )
 from audio_deepfake_detector.src.generalization_diagnostics import transform_audio
 changed=[]
 for variant in ('trim_pauses','volume','noise','format'):
  with tempfile.TemporaryDirectory() as temp:
   modified=files.copy()
   for i,row in enumerate(tqdm(files.itertuples(index=False),total=len(files),desc=variant)):
    x,rate=decode_audio(Path(row.audio_path));y=transform_audio(x,rate,variant,seed=seed.value)
    path=Path(temp)/f'{i:02}.flac';sf.write(path,y,rate,format='FLAC',subtype='PCM_16')
    modified.loc[modified.index[i],'audio_path']=str(path)
    modified.loc[modified.index[i],'sha256']=_sha256(path)
   outcome=evaluate_external(modified,checkpoint,prepared/'preprocessing_config.json',batch_size=8,git_commit=commit)
   merged=source[['source_file_id','fake_score']].merge(outcome['files'][['source_file_id','fake_score']],on='source_file_id',suffixes=('_original','_modified'))
   merged['variant']=variant;merged['delta']=merged.fake_score_modified-merged.fake_score_original
   merged['prediction_changed']=(merged.fake_score_original>=threshold)!=(merged.fake_score_modified>=threshold)
   changed.append(merged)
 changes=pd.concat(changed,ignore_index=True)
 display(changes.groupby('variant')[['delta','prediction_changed']].mean())
 print('Последний неполный сегмент: отдельная описательная проверка, без пересчёта основной метрики')

In [ ]:
# 10. Сравнение доменов
RUN_DOMAINS=False
if RUN_DOMAINS:
 if 'features' not in globals(): raise RuntimeError('Сначала ячейка 5')
 comparison=features.groupby(['domain','label']).agg(n=('id','size'),rms=('rms','median'),silence=('silence_fraction','median'),flatness=('flatness','median'),centroid=('centroid_hz','median'))
 display(comparison)
 print('Различие технических признаков — возможная связь, не доказательство причины ошибки CNN.')

In [ ]:
# 11. Итоговые выводы после анализа
RUN_CONCLUSIONS=False
if RUN_CONCLUSIONS:
 print('Подтверждено: порог, метки, checkpoint, Mel и агрегация согласованы; внешний тест 2/8.')
 print('Гипотезы: доменный сдвиг, условия записи, тишина, кодирование и отсутствие RVC в обучении.')
 print('После просмотра внешнего теста для новых решений потребуется другая независимая финальная выборка.')

In [ ]:
# 12. Сохранить обезличенную диагностику отдельным запуском
SAVE_DIAGNOSTICS=False
if SAVE_DIAGNOSTICS:
 from datetime import datetime
 from uuid import uuid4

 from audio_deepfake_detector.src.generalization_diagnostics import save_diagnostics
 if 'features' not in globals(): raise RuntimeError('Сначала выполните ячейку 5')
 output=base/'diagnostics_v1/baseline_cached_v1'/('run_'+datetime.now(UTC).strftime('%Y%m%d_%H%M%S')+'_'+uuid4().hex[:8])
 aliases={v:'Файл '+str(i+1) for i,v in enumerate(sorted(source.source_file_id))}
 safe_scores=segments[['source_file_id','segment_id','fake_score','true_label','predicted_label']].copy()
 safe_scores.source_file_id=safe_scores.source_file_id.map(aliases)
 safe_scores.segment_id=[aliases[id]+' / сегмент '+str(i) for id,i in zip(segments.source_file_id,segments.groupby('source_file_id').cumcount())]
 safe_changes=changes.copy() if 'changes' in globals() else pd.DataFrame()
 if not safe_changes.empty:safe_changes.source_file_id=safe_changes.source_file_id.map(aliases)
 cfg={'git_commit':commit,'checkpoint_sha256':_sha256(checkpoint),'source_run':'run_20261127_165511_c1b6379c','seed':seed.value,'sample_size_per_label':sample_size.value,'threshold':threshold,'mel_config':saved['mel_config']}
 domain_rows=features.groupby(['domain','label']).agg(n=('id','size'),median_rms=('rms','median'),median_silence=('silence_fraction','median'),median_flatness=('flatness','median')).reset_index()
 lines=['# Исследование причин снижения качества классификации на внешних аудиозаписях','',
        'Подтверждено: совместимость checkpoint, меток, Mel и агрегации. Внешняя Accuracy=0.25 (2/8).','',
        '## Технические признаки (медианы)','',
        '| Домен | Класс | N | RMS | Доля тишины | Flatness |',
        '|---|---|---:|---:|---:|---:|']
 for row in domain_rows.itertuples(index=False):
  lines.append(f'| {row.domain} | {row.label} | {row.n} | {row.median_rms:.5f} | {row.median_silence:.5f} | {row.median_flatness:.5f} |')
 if not safe_changes.empty:
  lines.extend(['','## Диагностические преобразования','',
                '| Преобразование | Среднее изменение FAKE-score | Смена класса |',
                '|---|---:|---:|'])
  for row in safe_changes.groupby('variant').agg(delta=('delta','mean'),changed=('prediction_changed','sum')).reset_index().itertuples(index=False):
   lines.append(f'| {row.variant} | {row.delta:.5f} | {row.changed} |')
 lines.extend(['','Различия признаков и изменения score не доказывают причину ошибок CNN.',
               'После использования этих восьми записей для анализа нужен новый независимый финальный тест.'])
 report='\n'.join(lines)+'\n'
 save_diagnostics(output,features,safe_scores,safe_changes,cfg,report)
 print('Результаты:',output)
 print('Проверьте diagnostic_report.md перед добавлением в журнал ВКР.')